# Sensi NLP — Étape 1 : Génération du dataset synthétique

**Objectif** : Construire un dataset de paires `(glosses LSF → phrase française)` à partir des 18 mots du vocabulaire Sensi.

Ce dataset servira à fine-tuner un modèle de type mBART ou similar.

**Pipeline** :
```
18 glosses → combinaisons → templates → dataset CSV/JSON
```

## 1. Vocabulaire Sensi

In [1]:
# Les 20 glosses du modèle LSTM Sensi
VOCABULARY = [
    "AIDER",
    "AMELIORER",
    "AMI",
    "AUJOURD_HUI",
    "BONJOUR",
    "COMMUNIQUER",
    "ENTENDANTS",
    "CONTENT",
    "JE_SUIS",
    "JE_VEUX",
    "LANGUE_DES_SIGNES",
    "MERCI",
    "OUTIL_POINTAGE",   # deixis : désigne un outil / objet
    "OUTIL",
    "PRESENTER",
    "PROJET",
    "SOURD_POINTAGE",   # deixis : désigne une personne malentendante
    "SOURD",            # → "malentendant(e)(s)" dans les phrases
    "TRADUCTION",       # → "traduire" / "traduction" selon contexte
    "VOCAL",            # → "vocal(e)" / "vocalement"
]

print(f"{len(VOCABULARY)} glosses dans le vocabulaire Sensi")
print(VOCABULARY)

20 glosses dans le vocabulaire Sensi
['AIDER', 'AMELIORER', 'AMI', 'AUJOURD_HUI', 'BONJOUR', 'COMMUNIQUER', 'ENTENDANTS', 'CONTENT', 'JE_SUIS', 'JE_VEUX', 'LANGUE_DES_SIGNES', 'MERCI', 'OUTIL_POINTAGE', 'OUTIL', 'PRESENTER', 'PROJET', 'SOURD_POINTAGE', 'SOURD', 'TRADUCTION', 'VOCAL']


## 2. Templates manuels

On définit des structures de phrases courantes en LSF avec leur traduction naturelle en français.

**Logique LSF** :
- Pas d'articles (le, la, un...)
- Pas d'auxiliaires en général
- Le temps est souvent implicite ou marqué par un adverbe (AUJOURD_HUI)
- Les pointages (OUTIL_POINTAGE, SOURD_POINTAGE) remplacent les pronoms

Chaque template est un tuple `(glosses, phrase_française)`.

In [2]:
MANUAL_PAIRS = [
    # --- Salutations ---
    (["BONJOUR"],
     "Bonjour."),

    (["MERCI"],
     "Merci."),

    (["BONJOUR", "MERCI"],
     "Bonjour et merci."),

    # --- Présentations du projet ---
    (["BONJOUR", "JE_SUIS", "CONTENT", "PRESENTER", "PROJET"],
     "Bonjour, je suis content de vous présenter ce projet."),

    (["BONJOUR", "AUJOURD_HUI", "PRESENTER", "PROJET"],
     "Bonjour, aujourd'hui nous allons vous présenter notre projet."),

    (["AUJOURD_HUI", "PRESENTER", "PROJET", "LANGUE_DES_SIGNES"],
     "Aujourd'hui, nous vous présentons notre projet de langue des signes."),

    (["JE_VEUX", "PRESENTER", "PROJET"],
     "Je veux vous présenter ce projet."),

    (["JE_SUIS", "CONTENT", "PRESENTER", "PROJET"],
     "Je suis content de vous présenter ce projet."),

    (["BONJOUR", "AUJOURD_HUI", "PRESENTER", "PROJET", "LANGUE_DES_SIGNES"],
     "Bonjour, aujourd'hui nous vous présentons notre projet en langue des signes."),

    (["JE_SUIS", "CONTENT", "AUJOURD_HUI", "PRESENTER", "PROJET"],
     "Je suis content de vous présenter notre projet aujourd'hui."),

    # --- Description du projet ---
    (["PROJET", "OUTIL", "TRADUCTION", "LANGUE_DES_SIGNES"],
     "Notre projet est un outil de traduction de la langue des signes."),

    (["PROJET", "OUTIL", "TRADUCTION", "LANGUE_DES_SIGNES", "VOCAL"],
     "Notre projet est un outil qui traduit la langue des signes en français vocal."),

    (["OUTIL_POINTAGE", "TRADUCTION", "LANGUE_DES_SIGNES", "VOCAL"],
     "Cet outil traduit la langue des signes en français vocal."),

    (["PROJET", "AIDER", "COMMUNIQUER", "SOURD", "ENTENDANTS"],
     "Ce projet aide les personnes malentendantes à communiquer avec les entendants."),

    (["PROJET", "OUTIL", "AIDER", "COMMUNIQUER", "SOURD", "ENTENDANTS"],
     "Notre projet est un outil qui aide les personnes malentendantes à communiquer avec les entendants."),

    (["OUTIL_POINTAGE", "AIDER", "COMMUNIQUER", "SOURD", "ENTENDANTS"],
     "Cet outil aide les personnes malentendantes à communiquer avec les entendants."),

    (["OUTIL_POINTAGE", "TRADUCTION", "LANGUE_DES_SIGNES"],
     "Cet outil traduit la langue des signes."),

    (["OUTIL_POINTAGE", "AIDER", "COMMUNIQUER"],
     "Cet outil aide à communiquer."),

    # --- Améliorer (dans le contexte du projet) ---
    (["PROJET", "AMELIORER", "COMMUNIQUER", "SOURD", "ENTENDANTS"],
     "Notre projet améliore la communication entre les personnes malentendantes et les entendants."),

    (["OUTIL_POINTAGE", "AMELIORER", "TRADUCTION", "LANGUE_DES_SIGNES"],
     "Cet outil améliore la traduction de la langue des signes."),

    (["JE_VEUX", "AMELIORER", "TRADUCTION", "VOCAL"],
     "Je veux améliorer la traduction vocale."),

    # --- Traduction et vocal ---
    (["TRADUCTION", "LANGUE_DES_SIGNES", "VOCAL"],
     "La traduction de la langue des signes est vocale."),

    (["JE_VEUX", "TRADUCTION", "LANGUE_DES_SIGNES"],
     "Je veux traduire la langue des signes."),

    (["LANGUE_DES_SIGNES", "TRADUCTION", "VOCAL"],
     "La langue des signes est traduite vocalement."),

    (["PROJET", "TRADUCTION", "VOCAL", "SOURD", "ENTENDANTS"],
     "Notre projet traduit vocalement les signes des personnes malentendantes pour les entendants."),

    (["JE_VEUX", "COMMUNIQUER", "VOCAL", "ENTENDANTS"],
     "Je veux communiquer vocalement avec les entendants."),

    (["OUTIL_POINTAGE", "TRADUCTION", "VOCAL", "LANGUE_DES_SIGNES"],
     "Cet outil traduit vocalement la langue des signes."),

    # --- Langue des signes et communication ---
    (["LANGUE_DES_SIGNES", "OUTIL", "COMMUNIQUER"],
     "La langue des signes est un outil pour communiquer."),

    (["JE_VEUX", "COMMUNIQUER", "ENTENDANTS"],
     "Je veux communiquer avec les entendants."),

    (["LANGUE_DES_SIGNES", "COMMUNIQUER", "SOURD", "ENTENDANTS"],
     "La langue des signes permet aux personnes malentendantes de communiquer avec les entendants."),

    (["JE_SUIS", "SOURD", "COMMUNIQUER", "LANGUE_DES_SIGNES"],
     "Je suis malentendant(e) et je communique en langue des signes."),

    (["JE_SUIS", "SOURD", "JE_VEUX", "COMMUNIQUER", "ENTENDANTS"],
     "Je suis malentendant(e) et je veux communiquer avec les entendants."),

    # --- Aider ---
    (["JE_VEUX", "AIDER", "SOURD", "COMMUNIQUER", "ENTENDANTS"],
     "Je veux aider les personnes malentendantes à communiquer avec les entendants."),

    (["JE_VEUX", "AIDER", "ENTENDANTS", "COMMUNIQUER", "SOURD"],
     "Je veux aider les entendants à communiquer avec les personnes malentendantes."),

    (["PROJET", "AIDER", "SOURD"],
     "Ce projet aide les personnes malentendantes."),

    (["OUTIL_POINTAGE", "AIDER", "SOURD", "COMMUNIQUER"],
     "Cet outil aide les personnes malentendantes à communiquer."),

    # --- Pointages (deixis) ---
    (["SOURD_POINTAGE", "JE_VEUX", "COMMUNIQUER"],
     "Cette personne malentendante veut communiquer."),

    (["SOURD_POINTAGE", "AMI", "ENTENDANTS"],
     "Cette personne malentendante a des amis entendants."),

    (["SOURD_POINTAGE", "COMMUNIQUER", "LANGUE_DES_SIGNES"],
     "Cette personne malentendante communique en langue des signes."),

    (["SOURD_POINTAGE", "TRADUCTION", "VOCAL"],
     "Cette personne malentendante utilise la traduction vocale."),

    # --- Amis ---
    (["AMI", "ENTENDANTS", "COMMUNIQUER"],
     "Mes amis entendants communiquent avec moi."),

    (["JE_SUIS", "CONTENT", "AMI"],
     "Je suis content d'avoir des amis."),

    (["MERCI", "AMI", "AIDER", "PROJET"],
     "Merci à mes amis qui ont aidé ce projet."),

    # --- Remerciements finaux ---
    (["MERCI", "AUJOURD_HUI", "PRESENTER", "PROJET"],
     "Merci de nous avoir écoutés présenter notre projet aujourd'hui."),

    (["JE_SUIS", "CONTENT", "COMMUNIQUER", "LANGUE_DES_SIGNES"],
     "Je suis content de communiquer en langue des signes."),

    (["AUJOURD_HUI", "JE_SUIS", "CONTENT", "PRESENTER"],
     "Aujourd'hui je suis content de vous présenter."),

    (["MERCI", "CONTENT", "PRESENTER", "PROJET"],
     "Merci, je suis content d'avoir présenté ce projet."),
]

print(f"{len(MANUAL_PAIRS)} paires manuelles créées")

47 paires manuelles créées


## 3. Vérification de la couverture du vocabulaire

In [3]:
from collections import Counter

# Compter l'utilisation de chaque gloss
gloss_counts = Counter()
for glosses, _ in MANUAL_PAIRS:
    for g in glosses:
        gloss_counts[g] += 1

print("Couverture du vocabulaire :")
print("-" * 40)
for gloss in VOCABULARY:
    count = gloss_counts.get(gloss, 0)
    bar = "█" * count
    status = "✓" if count > 0 else "✗ ABSENT"
    print(f"{status:10} {gloss:25} {bar} ({count}x)")

covered = sum(1 for g in VOCABULARY if gloss_counts.get(g, 0) > 0)
print(f"\nCouverture : {covered}/{len(VOCABULARY)} glosses utilisés")

Couverture du vocabulaire :
----------------------------------------
✓          AIDER                     █████████ (9x)
✓          AMELIORER                 ███ (3x)
✓          AMI                       ████ (4x)
✓          AUJOURD_HUI               ██████ (6x)
✓          BONJOUR                   █████ (5x)
✓          COMMUNIQUER               ██████████████████ (18x)
✓          ENTENDANTS                █████████████ (13x)
✓          CONTENT                   ███████ (7x)
✓          JE_SUIS                   ████████ (8x)
✓          JE_VEUX                   █████████ (9x)
✓          LANGUE_DES_SIGNES         ████████████████ (16x)
✓          MERCI                     █████ (5x)
✓          OUTIL_POINTAGE            ███████ (7x)
✓          OUTIL                     ████ (4x)
✓          PRESENTER                 ██████████ (10x)
✓          PROJET                    █████████████████ (17x)
✓          SOURD_POINTAGE            ████ (4x)
✓          SOURD                     ████████████ 

## 4. Génération augmentée par permutation

On augmente le dataset avec des variantes :
- Ajout de salutations en début/fin
- Ajout de `AUJOURD_HUI` en contexte temporel
- Variations sur `JE_SUIS` / `JE_VEUX`

In [4]:
# Augmentation simple : préfixes et suffixes courants en LSF
AUGMENTED_PAIRS = list(MANUAL_PAIRS)  # copie

# Règles d'augmentation
def add_bonjour(glosses, phrase):
    """Ajoute BONJOUR en début si pas déjà présent."""
    if glosses[0] != "BONJOUR":
        return ["BONJOUR"] + glosses, "Bonjour, " + phrase[0].lower() + phrase[1:]
    return None

def add_merci(glosses, phrase):
    """Ajoute MERCI en fin si pas déjà présent."""
    if glosses[-1] != "MERCI":
        return glosses + ["MERCI"], phrase.rstrip(".") + ", merci."
    return None

def add_aujourd_hui(glosses, phrase):
    """Ajoute AUJOURD_HUI en début si phrase d'action."""
    if "AUJOURD_HUI" not in glosses and len(glosses) >= 2:
        new_phrase = "Aujourd'hui, " + phrase[0].lower() + phrase[1:]
        # Évite le doublon si la phrase commence déjà par Aujourd'hui
        if not phrase.startswith("Aujourd"):
            return ["AUJOURD_HUI"] + glosses, new_phrase
    return None

# Appliquer les augmentations
augmented_count = 0
for glosses, phrase in MANUAL_PAIRS:
    for fn in [add_bonjour, add_merci, add_aujourd_hui]:
        result = fn(glosses, phrase)
        if result:
            new_glosses, new_phrase = result
            # Éviter les doublons
            if (new_glosses, new_phrase) not in AUGMENTED_PAIRS:
                AUGMENTED_PAIRS.append((new_glosses, new_phrase))
                augmented_count += 1

print(f"Paires originales  : {len(MANUAL_PAIRS)}")
print(f"Paires augmentées  : {augmented_count}")
print(f"Dataset total      : {len(AUGMENTED_PAIRS)}")

Paires originales  : 47
Paires augmentées  : 124
Dataset total      : 171


## 5. Mise en forme du dataset

In [5]:
import pandas as pd

# Construire le DataFrame
records = []
for glosses, phrase in AUGMENTED_PAIRS:
    records.append({
        "glosses": " ".join(glosses),          # "BONJOUR JE_SUIS CONTENT"
        "glosses_list": glosses,               # ["BONJOUR", "JE_SUIS", "CONTENT"]
        "phrase": phrase,                       # "Bonjour, je suis content."
        "n_glosses": len(glosses),
    })

df = pd.DataFrame(records)
df = df.drop_duplicates(subset=["glosses"]).reset_index(drop=True)

print(f"Dataset final : {len(df)} exemples uniques")
print(f"\nDistribution du nombre de glosses :")
print(df["n_glosses"].value_counts().sort_index())
print(f"\n--- Aperçu ---")
df.head(10)

Dataset final : 169 exemples uniques

Distribution du nombre de glosses :
n_glosses
1     2
2     1
3    16
4    62
5    55
6    30
7     3
Name: count, dtype: int64

--- Aperçu ---


,glosses,glosses_list,phrase,n_glosses
0,BONJOUR,[BONJOUR],Bonjour.,1
1,MERCI,[MERCI],Merci.,1
2,BONJOUR MERCI,"[BONJOUR, MERCI]",Bonjour et merci.,2
3,BONJOUR JE_SUIS CONTENT PRESENTER PROJET,"[BONJOUR, JE_SUIS, CONTENT, PRESENTER, PROJET]","Bonjour, je suis content de vous présenter ce ...",5
4,BONJOUR AUJOURD_HUI PRESENTER PROJET,"[BONJOUR, AUJOURD_HUI, PRESENTER, PROJET]","Bonjour, aujourd'hui nous allons vous présente...",4
5,AUJOURD_HUI PRESENTER PROJET LANGUE_DES_SIGNES,"[AUJOURD_HUI, PRESENTER, PROJET, LANGUE_DES_SI...","Aujourd'hui, nous vous présentons notre projet...",4
6,JE_VEUX PRESENTER PROJET,"[JE_VEUX, PRESENTER, PROJET]",Je veux vous présenter ce projet.,3
7,JE_SUIS CONTENT PRESENTER PROJET,"[JE_SUIS, CONTENT, PRESENTER, PROJET]",Je suis content de vous présenter ce projet.,4
8,BONJOUR AUJOURD_HUI PRESENTER PROJET LANGUE_DE...,"[BONJOUR, AUJOURD_HUI, PRESENTER, PROJET, LANG...","Bonjour, aujourd'hui nous vous présentons notr...",5
9,JE_SUIS CONTENT AUJOURD_HUI PRESENTER PROJET,"[JE_SUIS, CONTENT, AUJOURD_HUI, PRESENTER, PRO...",Je suis content de vous présenter notre projet...,5


In [6]:
# Affichage lisible de quelques exemples
print("=" * 60)
print("EXEMPLES DU DATASET")
print("=" * 60)
for _, row in df.sample(10, random_state=42).iterrows():
    print(f"Glosses : {row['glosses']}")
    print(f"Phrase  : {row['phrase']}")
    print("-" * 60)

EXEMPLES DU DATASET
Glosses : BONJOUR SOURD_POINTAGE JE_VEUX COMMUNIQUER
Phrase  : Bonjour, cette personne malentendante veut communiquer.
------------------------------------------------------------
Glosses : JE_SUIS SOURD COMMUNIQUER LANGUE_DES_SIGNES
Phrase  : Je suis malentendant(e) et je communique en langue des signes.
------------------------------------------------------------
Glosses : AUJOURD_HUI LANGUE_DES_SIGNES COMMUNIQUER SOURD ENTENDANTS
Phrase  : Aujourd'hui, la langue des signes permet aux personnes malentendantes de communiquer avec les entendants.
------------------------------------------------------------
Glosses : LANGUE_DES_SIGNES COMMUNIQUER SOURD ENTENDANTS
Phrase  : La langue des signes permet aux personnes malentendantes de communiquer avec les entendants.
------------------------------------------------------------
Glosses : AUJOURD_HUI SOURD_POINTAGE AMI ENTENDANTS
Phrase  : Aujourd'hui, cette personne malentendante a des amis entendants.
------------------

## 6. Sauvegarde

In [7]:
import json
from pathlib import Path

OUTPUT_DIR = Path("data/nlp")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Sauvegarde CSV
csv_path = OUTPUT_DIR / "sensi_dataset.csv"
df[["glosses", "phrase", "n_glosses"]].to_csv(csv_path, index=False)
print(f"✓ CSV sauvegardé : {csv_path}")

# Sauvegarde JSON (format HuggingFace datasets)
json_path = OUTPUT_DIR / "sensi_dataset.json"
records_json = [
    {"source": row["glosses"], "target": row["phrase"]}
    for _, row in df.iterrows()
]
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(records_json, f, ensure_ascii=False, indent=2)
print(f"✓ JSON sauvegardé : {json_path}")

print(f"\nDataset prêt : {len(df)} paires glosses → phrases")

✓ CSV sauvegardé : data/nlp/sensi_dataset.csv
✓ JSON sauvegardé : data/nlp/sensi_dataset.json

Dataset prêt : 169 paires glosses → phrases


## 7. Statistiques finales

In [8]:
print("=" * 50)
print("BILAN DATASET SENSI NLP")
print("=" * 50)
print(f"Nombre de paires         : {len(df)}")
print(f"Longueur min (glosses)   : {df['n_glosses'].min()}")
print(f"Longueur max (glosses)   : {df['n_glosses'].max()}")
print(f"Longueur moyenne         : {df['n_glosses'].mean():.1f}")
print(f"Vocabulaire couvert      : {len(set(' '.join(df['glosses']).split()))} / {len(VOCABULARY)} glosses")
print()
print("Prochaine étape : fine-tuning mBART ou CamemBERT sur ce dataset")

BILAN DATASET SENSI NLP
Nombre de paires         : 169
Longueur min (glosses)   : 1
Longueur max (glosses)   : 7
Longueur moyenne         : 4.6
Vocabulaire couvert      : 20 / 20 glosses

Prochaine étape : fine-tuning mBART ou CamemBERT sur ce dataset


## Model moussaKam/barthez

In [9]:
# !pip install transformers datasets sentencepiece sacrebleu --quiet
# !pip install transformers==4.40.0 --quiet
# !pip install torch torchvision torchaudio --quiet
# !pip install accelerate -U --quiet

In [10]:
import json
import pandas as pd
from pathlib import Path

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)
from datasets import Dataset

/Users/franckabeille/.pyenv/versions/3.10.6/envs/sensi/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Chargement modèle

In [11]:
import torch
from transformers import CamembertTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "moussaKam/barthez"

# Device : MPS (GPU Apple Silicon) si dispo, sinon CPU
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device : {device}")

# Chargement tokenizer et modèle
tokenizer = CamembertTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model = model.to(device)

print(f"Modèle chargé : {MODEL_NAME}")
print(f"Paramètres    : {model.num_parameters():,}")

Device : mps


/Users/franckabeille/.pyenv/versions/3.10.6/envs/sensi/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'BarthezTokenizer'. 
The class this function is called from is 'CamembertTokenizer'.


Modèle chargé : moussaKam/barthez
Paramètres    : 139,221,504


## Chargement dataset

In [12]:
# Chargement du dataset généré à l'étape 1
dataset_path = Path("data/nlp/sensi_dataset.json")

with open(dataset_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# Conversion en Dataset HuggingFace
dataset = Dataset.from_list(data)

# Split train / validation (90% / 10%)
dataset = dataset.train_test_split(test_size=0.1, seed=42)

print(f"Train      : {len(dataset['train'])} exemples")
print(f"Validation : {len(dataset['test'])} exemples")
print(f"\n--- Aperçu ---")
print(dataset['train'][0])

Train      : 152 exemples
Validation : 17 exemples

--- Aperçu ---
{'source': 'AUJOURD_HUI JE_VEUX COMMUNIQUER VOCAL ENTENDANTS', 'target': "Aujourd'hui, je veux communiquer vocalement avec les entendants."}


## Tokenisation

In [13]:
# Longueur maximale des séquences
MAX_INPUT_LENGTH  = 64   # glosses (courtes, 2-6 mots)
MAX_TARGET_LENGTH = 128  # phrases françaises (plus longues)

def tokenize(examples):
    # Tokenisation des glosses (entrée)
    model_inputs = tokenizer(
        examples["source"],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding=False,
    )
    # Tokenisation des phrases françaises (cible)
    labels = tokenizer(
        examples["target"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding=False,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Application sur tout le dataset
tokenized_dataset = dataset.map(
    tokenize,
    batched=True,
    remove_columns=["source", "target"],
)

print("Tokenisation terminée")
print(f"Features : {tokenized_dataset['train'].features}")
print(f"\nExemple tokenisé :")
print(tokenized_dataset['train'][0])

Map: 100%|██████████| 17/17 [00:00<00:00, 5165.40 examples/s]

Tokenisation terminée
Features : {'input_ids': List(Value('int32')), 'attention_mask': List(Value('int8')), 'labels': List(Value('int64'))}

Exemple tokenisé :
{'input_ids': [50003, 4099, 36233, 49951, 7, 49976, 5105, 9942, 7, 5920, 8624, 13627, 49948, 45925, 223, 3828, 1713, 2867, 49953, 9946, 16233, 6], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [50003, 3466, 49932, 880, 49925, 239, 3675, 7344, 8185, 408, 171, 52, 4400, 268, 49926, 6]}


## Data collator et training args

In [50]:
# Data collator — gère le padding dynamique par batch
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,  # -100 = ignoré dans le calcul de la loss
)

# Arguments d'entraînement
training_args = Seq2SeqTrainingArguments(
    output_dir="models/barthez_sensi_final",
    num_train_epochs=20,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=5e-5,
    warmup_steps=10,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    predict_with_generate=True,
    fp16=False,       # désactivé — non supporté sur MPS
    # use_mps_device=True,
    logging_steps=10,
    seed=42,
)

print("Configuration prête")
print(f"Epochs         : {training_args.num_train_epochs}")
print(f"Batch size     : {training_args.per_device_train_batch_size}")
print(f"Learning rate  : {training_args.learning_rate}")

Configuration prête
Epochs         : 20
Batch size     : 8
Learning rate  : 5e-05


## Entraînement

In [38]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# Lancement de l'entraînement
trainer.train()

  0%|          | 0/380 [00:00<?, ?it/s]/Users/franckabeille/.pyenv/versions/3.10.6/envs/sensi/lib/python3.10/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
  3%|▎         | 10/380 [00:05<03:19,  1.85it/s]

{'loss': 0.1637, 'grad_norm': 6.1860527992248535, 'learning_rate': 5e-05, 'epoch': 0.53}


  5%|▌         | 19/380 [00:10<03:16,  1.84it/s]Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'forced_eos_token_id': 2}
Your generation config was originally created from the model config, but the model config has changed since then. Unless you pass the `generation_config` argument to this model's `generate` calls, they will revert to the legacy behavior where the base `generate` parameterization is loaded from the model config instead. To avoid this behavior and this warning, we recommend you to overwrite the generation config model attribute before calling the model's `save_pretrained`, preferably also removing any generation kwargs from the model config. This warning will be raised to an exception in v4.41.


{'eval_loss': 0.24616308510303497, 'eval_runtime': 0.4945, 'eval_samples_per_second': 34.376, 'eval_steps_per_second': 6.066, 'epoch': 1.0}


/Users/franckabeille/.pyenv/versions/3.10.6/envs/sensi/lib/python3.10/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
  6%|▌         | 21/380 [00:13<05:46,  1.04it/s]

{'loss': 0.2907, 'grad_norm': 5.334789276123047, 'learning_rate': 4.8648648648648654e-05, 'epoch': 1.05}


  7%|▋         | 27/380 [00:16<03:25,  1.72it/s]

KeyboardInterrupt: 

### Nettoyage des opimizer.pt

In [ ]:
import glob
import os

# Suppression des optimizer.pt après entraînement
for f in glob.glob("models/barthez_sensi/**/optimizer.pt", recursive=True):
    os.remove(f)
    print(f"Supprimé : {f}")

## Inférence

In [ ]:
import re
from transformers import pipeline

# Définition du generator
generator = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    device=device,
)

def generate_phrase(glosses: str) -> str:
    """Convertit une séquence de glosses en phrase française."""
    result = generator(
        glosses,
        max_new_tokens=40,
        num_beams=4,
        early_stopping=True,
        no_repeat_ngram_size=3,
        repetition_penalty=2.0,
    )
    phrase = result[0]["generated_text"]
    match = re.search(r'[.!?]', phrase)
    if match:
        phrase = phrase[:match.end()]
    return phrase.strip()

test_glosses = [
    "BONJOUR JE_SUIS CONTENT PRESENTER PROJET",
    "JE_VEUX AIDER SOURD COMMUNIQUER ENTENDANTS",  # SOURDE → SOURD
    "LANGUE_DES_SIGNES OUTIL COMMUNIQUER",
    "AUJOURD_HUI PRESENTER PROJET LANGUE_DES_SIGNES",
    "MERCI AMI AIDER PROJET",
    "PROJET OUTIL TRADUCTION LANGUE_DES_SIGNES VOCAL",
    "OUTIL_POINTAGE TRADUCTION VOCAL LANGUE_DES_SIGNES",
    "SOURD_POINTAGE TRADUCTION VOCAL",
]

print("=== Tests d'inférence ===\n")
for glosses in test_glosses:
    phrase = generate_phrase(glosses)
    print(f"Glosses : {glosses}")
    print(f"Phrase  : {phrase}")
    print("-" * 50)

=== Tests d'inférence ===



/Users/franckabeille/.pyenv/versions/3.10.6/envs/sensi/lib/python3.10/site-packages/transformers/generation/utils.py:1256: UserWarning: You have modified the pretrained model configuration to control generation. This is a deprecated strategy to control generation and will be removed soon, in a future version. Please use and modify the model generation configuration (see https://huggingface.co/docs/transformers/generation_strategies#default-text-generation-configuration )
  warnings.warn(


Glosses : BONJOUR JE_SUIS CONTENT PRESENTER PROJET
Phrase  : Bonjour, je suis content de vous présenter ce projet.
--------------------------------------------------
Glosses : JE_VEUX AIDER SOURD COMMUNIQUER ENTENDANTS
Phrase  : Je veux aider les entendants à communiquer avec les personnes malentendantes.
--------------------------------------------------
Glosses : LANGUE_DES_SIGNES OUTIL COMMUNIQUER
Phrase  : La langue des signes est un outil pour communiquer.
--------------------------------------------------
Glosses : AUJOURD_HUI PRESENTER PROJET LANGUE_DES_SIGNES
Phrase  : Aujourd'hui, nous vous présentons notre projet de langue des signes.
--------------------------------------------------
Glosses : MERCI AMI AIDER PROJET
Phrase  : Merci à mes amis qui ont aidé ce projet.
--------------------------------------------------
Glosses : PROJET OUTIL TRADUCTION LANGUE_DES_SIGNES VOCAL
Phrase  : Notre projet est un outil de traduction de la langue des signes en français vocal.
----------

## Sauvegarde

In [ ]:
from pathlib import Path
from transformers import GenerationConfig

SAVE_DIR = Path("models/barthez_sensi_final")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Suppression des paramètres de génération de la config du modèle
for param in ["early_stopping", "num_beams", "no_repeat_ngram_size", "forced_eos_token_id"]:
    if hasattr(model.config, param):
        delattr(model.config, param)

# Sauvegarde des paramètres de génération dans un fichier dédié
generation_config = GenerationConfig(
    early_stopping=True,
    num_beams=4,
    no_repeat_ngram_size=3,
    max_new_tokens=40,
    repetition_penalty=2.0,
    forced_eos_token_id=2,
)
generation_config.save_pretrained(SAVE_DIR)

# Sauvegarde du modèle et du tokenizer
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print(f"✓ Modèle sauvegardé dans : {SAVE_DIR}")
for f in sorted(SAVE_DIR.iterdir()):
    size = f.stat().st_size / 1024 / 1024
    print(f"  {f.name:40} {size:.1f} Mo")

✓ Modèle sauvegardé dans : models/barthez_sensi_final
  added_tokens.json                        0.0 Mo
  config.json                              0.0 Mo
  generation_config.json                   0.0 Mo
  model.safetensors                        531.3 Mo
  sentencepiece.bpe.model                  1.1 Mo
  special_tokens_map.json                  0.0 Mo
  tokenizer_config.json                    0.0 Mo


# --- Rechargement du modèle (optionnel) ---

In [39]:
# À utiliser si le kernel a été redémarré et que le modèle est déjà entraîné.
# Remplace les cellules 3 à 7 dans ce cas.

model = AutoModelForSeq2SeqLM.from_pretrained("models/barthez_sensi_final")
model = model.to(device)
tokenizer = CamembertTokenizer.from_pretrained("models/barthez_sensi_final")

print("✓ Modèle rechargé depuis models/barthez_sensi_final")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


✓ Modèle rechargé depuis models/barthez_sensi_final


## Text-to-Speech

In [ ]:
# !pip install pyttsx3 --quiet

In [45]:
import pyttsx3

engine = pyttsx3.init()

# Recherche de la voix Thomas (français de France)
voices = engine.getProperty("voices")
thomas_voice = None
for voice in voices:
    if "thomas" in voice.name.lower() or "thomas" in voice.id.lower():
        thomas_voice = voice
        print(f"Voix trouvée : {voice.name} ({voice.id})")
        break

if thomas_voice:
    engine.setProperty("voice", thomas_voice.id)
else:
    print("Thomas non trouvé — voix disponibles :")
    for voice in voices:
        if "fr" in voice.id.lower() or "french" in voice.name.lower():
            print(f"  {voice.name} ({voice.id})")

# Paramètres
engine.setProperty("rate", 140)
engine.setProperty("volume", 1.0)

def speak(phrase: str):
    """Lit la phrase à voix haute."""
    print(f"🔊 {phrase}")
    engine.say(phrase)
    engine.runAndWait()

# Test
speak("Bonjour, je suis content de vous présenter ce projet.")

Voix trouvée : Thomas (com.apple.voice.compact.fr-FR.Thomas)
🔊 Bonjour, je suis content de vous présenter ce projet.


In [46]:
import subprocess

def speak(phrase: str):
    """Lit la phrase à voix haute via la commande say de macOS."""
    print(f"🔊 {phrase}")
    subprocess.run(["say", "-v", "Thomas", "-r", "140", phrase])

# Test
speak("Bonjour, je suis content de vous présenter ce projet.")

🔊 Bonjour, je suis content de vous présenter ce projet.


## Pipeline complet

In [41]:
def pipeline_sensi(lstm_output: list[str]) -> str:
    """
    Pipeline complet Sensi.
    Input  : liste de glosses prédites par le LSTM
    Output : phrase française lue vocalement
    """
    glosses_str = " ".join(lstm_output)
    print(f"Glosses  : {glosses_str}")

    phrase = generate_phrase(glosses_str)
    print(f"Phrase   : {phrase}")

    speak(phrase)
    return phrase


# Test
print("=" * 60)
print("PIPELINE SENSI — TEST COMPLET")
print("=" * 60)

tests = [
    ["BONJOUR", "JE_SUIS", "CONTENT", "PRESENTER", "PROJET"],
    ["PROJET", "OUTIL", "TRADUCTION", "LANGUE_DES_SIGNES", "VOCAL"],
    ["JE_VEUX", "AIDER", "SOURD", "COMMUNIQUER", "ENTENDANTS"],
    ["AUJOURD_HUI", "PRESENTER", "PROJET", "LANGUE_DES_SIGNES"],
    ["MERCI"],
]

for lstm_output in tests:
    print()
    pipeline_sensi(lstm_output)
    print("-" * 60)

PIPELINE SENSI — TEST COMPLET

Glosses  : BONJOUR JE_SUIS CONTENT PRESENTER PROJET
Phrase   : Bonjour, je suis content de vous présenter ce projet.
🔊 Bonjour, je suis content de vous présenter ce projet.
------------------------------------------------------------

Glosses  : PROJET OUTIL TRADUCTION LANGUE_DES_SIGNES VOCAL
Phrase   : Notre projet est un outil de traduction de la langue des signes en français vocal.
🔊 Notre projet est un outil de traduction de la langue des signes en français vocal.
------------------------------------------------------------

Glosses  : JE_VEUX AIDER SOURD COMMUNIQUER ENTENDANTS
Phrase   : Je veux aider les entendants à communiquer avec les personnes malentendantes.
🔊 Je veux aider les entendants à communiquer avec les personnes malentendantes.
------------------------------------------------------------

Glosses  : AUJOURD_HUI PRESENTER PROJET LANGUE_DES_SIGNES
Phrase   : Aujourd'hui, nous vous présentons notre projet de langue des signes.
🔊 Aujourd'h

In [51]:
pipeline_sensi(["JE_VEUX", "AIDER", "COMMUNIQUER", "ENTENDANTS", "SOURD"])

Glosses  : JE_VEUX AIDER COMMUNIQUER ENTENDANTS SOURD
Phrase   : Je veux aider les entendants à communiquer avec les personnes malentendantes.
🔊 Je veux aider les entendants à communiquer avec les personnes malentendantes.


'Je veux aider les entendants à communiquer avec les personnes malentendantes.'